# AlexNet (2012) — 새로 넣은 넷의 몫을 한 표에 놓는다Krizhevsky, Sutskever, Hinton 이 **새로 넣었다고 꼽은 넷**(ReLU · LRN · 겹치는 풀링 · 드롭아웃)을 CIFAR-10 에서 하나씩 켜고 끈다.| 실험 | 묻는 것 | 노트의 어느 자리에 닿는가 ||---|---|---|| A | **ReLU 가 tanh 보다 몇 배 빨리 학습 오차 25% 에 닿는가** | 13절 일곱 — 그림 1 의 「여섯 배」는 조건이 붙은 값이다 || B | LRN 을 켜고 끄면 얼마인가 | 13절 다섯 — 상수 넷의 탐색 범위가 없다. 논문의 CIFAR-10 자리는 13% 대 11% || C | 겹치는 풀링($z{=}3$) 대 안 겹치는 풀링($z{=}2$) | 6.4절 — 이득 −0.4 / −0.3 || **D** | **드롭아웃을 끄면 얼마인가** | **13절 여섯 — 넷 중 드롭아웃만 이득이 숫자로 없다** || **E** | **넷을 함께 끈 차이가 하나씩 끈 차이의 합과 같은가** | **13절 넷 — 둘 이상을 함께 끈 행이 논문에 없다** |---## 돌리기 전에 밝혀 두는 것 다섯**하나. ImageNet 이 아니라 CIFAR-10 이다.** 120만 장을 GPU 두 장으로 5~6 일 돌린 것을 다시 할 수 없다. 대신 **논문 자신이 자기 장치를 확인한 자리**를 쓴다 — 그림 1(ReLU)과 3.3절 각주(LRN 13%→11%)가 둘 다 CIFAR-10 네 층 망에서 나온 값이다. **그래도 −1.7 · −1.4 · −0.4 같은 논문의 ImageNet 값과 내 값을 같은 칸에 놓지 않는다.****둘. 논문의 CIFAR-10 망 구조를 모른다.** 3.3절 각주가 **자리가 모자라 적지 못한다**고 밝히고 코드로 대신한다. **그래서 이 노트북은 13% · 11% 를 재현하는 것이 아니다** — 내가 세운 망 안에서 같은 장치를 켜고 끄는 것이고, 비교는 그 안에서만 성립한다.**셋. GPU 둘로 쪼개기는 재지 않는다.** 장비가 한 장이고, 그 이득(−1.7)은 **메모리 3GB 라는 2012년의 제약**에서 나온 것이라 지금 재면 다른 것을 재게 된다.**넷. 데이터 늘리기를 켜지 않는다.** D 가 묻는 것이 **드롭아웃이 과적합을 얼마나 줄이는가**라서, 과적합을 먼저 줄여 놓으면 그 물음이 흐려진다.**다섯. 초기화만 논문 값을 기본으로 쓰지 않는다.** 논문의 $\sigma = 0.01$ 은 팬인이 훨씬 큰 층들을 위한 값이라 이 작은 망에서는 학습이 느리다(예비 실행에서 네 바퀴에 시험 오차 74% 대 He 초기화 55%). **He 초기화를 기본으로 두고, 논문 값 조건을 제거 실험의 한 행으로 함께 잰다.****판정 규칙.** seed 셋의 **변동 폭(최소~최대)을 먼저 적고, 그 폭보다 작은 차이로 순서를 매기지 않는다.** D 는 **학습 오차와 시험 오차의 간격**을 함께 적어 「과적합이 없었다」와 「드롭아웃이 안 들었다」를 가른다. E 는 **하나씩 끈 차이의 합**과 **넷을 함께 끈 차이**를 나란히 적고 그 둘의 차이를 변동 폭과 견준다.

In [ ]:
# ── 준비 ────────────────────────────────────────────────────────────────
import os, sys, time, math, platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

NOTEBOOK = "alexnet_2012.ipynb"
RUN_ID   = time.strftime("%Y%m%d_%H%M%S")
HERE     = os.getcwd()
RESULTS  = os.path.join(HERE, "results")
FIGURES  = os.path.join(HERE, "figures")
DATA     = os.path.join(HERE, "data")
for d in (RESULTS, FIGURES, DATA):
    os.makedirs(d, exist_ok=True)

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENV = {
    "run_id": RUN_ID, "notebook": NOTEBOOK,
    "python": sys.version.split()[0], "torch": torch.__version__,
    "numpy": np.__version__, "pandas": pd.__version__,
    "device": (torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu"),
    "platform": platform.platform(), "cwd": HERE,
}
for k, v in ENV.items():
    print("%-9s %s" % (k, v))


def save(df, name):
    """결과 CSV 에 재현 정보를 열로 붙여 저장한다."""
    df = df.copy()
    for k in ("run_id", "notebook", "python", "torch", "device"):
        df[k] = ENV[k]
    path = os.path.join(RESULTS, name)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("저장:", name, df.shape)
    return df


def span(v):
    """변동 폭을 '최소~최대' 로 적는다. 판정 규칙이 요구하는 것."""
    v = np.asarray(v, float)
    return "%.2f~%.2f" % (v.min(), v.max())


# ── 이 실행의 손잡이 ─────────────────────────────────────────────────────
N_TRAIN  = 20_000     # CIFAR-10 학습 50,000 장 중 일부만 쓴다
N_TEST   = 10_000     # 시험은 전부
EPOCHS   = 20
BATCH    = 128        # 논문 5절과 같다
MOMENTUM = 0.9        # 논문 5절
WDECAY   = 0.0005     # 논문 5절 — 「정칙화가 아니라 학습 오차를 줄인다」고 적은 값
LR       = 0.01       # 논문 5절의 출발값
LR_DROPS = (10, 16)   # 이 바퀴에서 10 으로 나눈다(논문은 검증 오차를 보고 손으로 내렸다)
SEEDS    = [0, 1, 2]
FAST     = False      # True 면 학습 장수만 절반으로 줄인다(축소 규칙)
if FAST:
    N_TRAIN = N_TRAIN // 2

print()
print("학습 %d 장 · 시험 %d 장 · %d 바퀴 · 묶음 %d · seed %s"
      % (N_TRAIN, N_TEST, EPOCHS, BATCH, SEEDS))
print("관성 %.1f · 가중치 감쇠 %.4f · 학습률 %.3f (바퀴 %s 에서 1/10)"
      % (MOMENTUM, WDECAY, LR, LR_DROPS))

## 0. 데이터 — CIFAR-10논문이 자기 장치를 확인한 자리와 같은 데이터다. **채널별 평균과 표준편차로 표준화만** 하고 데이터 늘리기는 켜지 않는다(위 「밝혀 두는 것 넷」).받지 못하면 **그 사실을 적고 멈춘다** — 합성 데이터로 바꿔치기하면 결과를 논문 이야기와 잇지 못한다.

In [ ]:
# ── 데이터 ──────────────────────────────────────────────────────────────
from torchvision import datasets

def load(train, n, seed=12345):
    ds = datasets.CIFAR10(DATA, train=train, download=True)
    X = torch.from_numpy(ds.data).float().permute(0, 3, 1, 2) / 255.0   # (N,3,32,32)
    y = torch.tensor(ds.targets, dtype=torch.long)
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g)[:n]
    return X[idx], y[idx]


Xtr, ytr = load(True,  N_TRAIN)
Xte, yte = load(False, N_TEST, seed=777)

MEAN = Xtr.mean(dim=(0, 2, 3), keepdim=True)
STD  = Xtr.std(dim=(0, 2, 3), keepdim=True)
Xtr = ((Xtr - MEAN) / STD).to(DEV)
Xte = ((Xte - MEAN) / STD).to(DEV)
ytr, yte = ytr.to(DEV), yte.to(DEV)

CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]
print("학습", tuple(Xtr.shape), "시험", tuple(Xte.shape))
print("채널 평균", [round(v, 3) for v in MEAN.flatten().tolist()],
      "· 표준편차", [round(v, 3) for v in STD.flatten().tolist()])
print("부류별 장수", torch.bincount(ytr, minlength=10).tolist())

## 1. 구현 — 켜고 끄는 넷을 손잡이로 뺀다망은 **학습되는 층 넷**이다 — 합성곱 셋(커널 32 · 32 · 64)과 완전연결 하나. 논문의 「네 층 망」을 이렇게 읽었다.손잡이 넷:- `act`: `"relu"` 또는 `"tanh"`- `lrn`: 켜면 합성곱 1 · 2 뒤에 국소 반응 정규화. 상수는 **논문 값 그대로** $k{=}2, n{=}5, \alpha{=}10^{-4}, \beta{=}0.75$- `pool_z`: 3 이면 겹치는 풀링($s{=}2$), 2 면 안 겹치는 풀링- `dropout`: 켜면 완전연결 앞에서 0.5- `init`: `"he"`(기본) 또는 `"paper"`(논문 5절의 $\sigma = 0.01$)**초기화만 논문 값을 기본으로 쓰지 않는다.** 논문의 $\sigma = 0.01$ 은 팬인이 수백에서 수천인 층들을 위한 값이라, 이 작은 망(첫 층 팬인 27)에서는 신호가 너무 작아 학습이 느리다. **그래서 He 초기화를 기본으로 두되, 논문 값으로 돌린 조건을 제거 실험의 한 행으로 함께 넣는다** — 바꾼 것을 말로만 적지 않고 재려는 것이다.**LRN 은 직접 구현한다.** PyTorch 의 `nn.LocalResponseNorm` 은 $k + \frac{\alpha}{n}\sum$ 로 **$\alpha$ 를 $n$ 으로 나누어 쓰는데**, 논문 식은 $k + \alpha\sum$ 다. 같은 $\alpha$ 를 넣으면 다섯 배 약한 정규화가 된다. **아래 검증 칸에서 이 차이를 숫자로 확인한다.**

In [ ]:
# ── 구현 ────────────────────────────────────────────────────────────────
class PaperLRN(nn.Module):
    """논문 식 그대로의 국소 반응 정규화 — b = a / (k + alpha * sum_n a^2)^beta."""
    def __init__(self, n=5, k=2.0, alpha=1e-4, beta=0.75):
        super().__init__()
        self.n, self.k, self.alpha, self.beta = n, k, alpha, beta

    def forward(self, x):
        pad = self.n // 2
        sq = x.pow(2).unsqueeze(1)                      # (N,1,C,H,W)
        sq = F.pad(sq, (0, 0, 0, 0, pad, pad))          # 채널 축에만 0 을 붙인다
        s = F.avg_pool3d(sq, (self.n, 1, 1), stride=1).squeeze(1) * self.n   # 평균 x n = 합
        return x / (self.k + self.alpha * s).pow(self.beta)


class Net(nn.Module):
    """학습되는 층 넷 — 합성곱 셋과 완전연결 하나."""
    def __init__(self, act="relu", lrn=True, pool_z=3, dropout=True, init="he"):
        super().__init__()
        self.act_name = act
        self.c1 = nn.Conv2d(3, 32, 3, padding=1)
        self.c2 = nn.Conv2d(32, 32, 5, padding=2)
        self.c3 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc = nn.Linear(64 * 4 * 4, 10)
        self.lrn = PaperLRN() if lrn else None
        self.pool = nn.MaxPool2d(pool_z, stride=2, padding=(1 if pool_z == 3 else 0))
        self.drop = nn.Dropout(0.5) if dropout else None
        for m in (self.c1, self.c2, self.c3, self.fc):
            if init == "paper":
                nn.init.normal_(m.weight, 0.0, 0.01)    # 논문 5절의 값 그대로
            else:
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            nn.init.zeros_(m.bias)
        nn.init.ones_(self.c2.bias)                     # 논문 5절 — ReLU 에 양의 입력을 준다
        nn.init.ones_(self.c3.bias)

    def f(self, x):
        return torch.relu(x) if self.act_name == "relu" else torch.tanh(x)

    def forward(self, x):
        x = self.f(self.c1(x))
        if self.lrn is not None:
            x = self.lrn(x)
        x = self.pool(x)
        x = self.f(self.c2(x))
        if self.lrn is not None:
            x = self.lrn(x)
        x = self.pool(x)
        x = self.f(self.c3(x))
        x = self.pool(x)
        x = x.flatten(1)
        if self.drop is not None:
            x = self.drop(x)
        return self.fc(x)


CONDS = {
    "기준 (넷 다 켬)":   dict(act="relu", lrn=True,  pool_z=3, dropout=True),
    "LRN 끔":            dict(act="relu", lrn=False, pool_z=3, dropout=True),
    "겹침 끔 (z=2)":     dict(act="relu", lrn=True,  pool_z=2, dropout=True),
    "드롭아웃 끔":       dict(act="relu", lrn=True,  pool_z=3, dropout=False),
    "ReLU 끔 (tanh)":    dict(act="tanh", lrn=True,  pool_z=3, dropout=True),
    "넷 다 끔":          dict(act="tanh", lrn=False, pool_z=2, dropout=False),
    "초기화 논문 값":    dict(act="relu", lrn=True,  pool_z=3, dropout=True, init="paper"),
}


@torch.no_grad()
def err_pct(model, X, y, bs=2000):
    model.eval()
    wrong = 0
    for i in range(0, X.shape[0], bs):
        wrong += (model(X[i:i + bs]).argmax(1) != y[i:i + bs]).sum().item()
    return 100.0 * wrong / X.shape[0]


def train(cfg, seed=0, epochs=None, lr=None, wdecay=None, log_rows=None, tag="",
          stop_at=None, max_updates=None):
    """seed 를 먼저 박고 망을 세운 뒤 배운다.

    stop_at 을 주면 **묶음 50 개의 달리는 학습 오차**가 그 값 아래로 내려간 순간의
    갱신 횟수를 돌려준다(실험 A 가 쓴다).
    """
    epochs = EPOCHS if epochs is None else epochs
    lr = LR if lr is None else lr
    wdecay = WDECAY if wdecay is None else wdecay
    torch.manual_seed(seed)
    model = Net(**cfg).to(DEV)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM, weight_decay=wdecay)
    n, t0, updates = Xtr.shape[0], time.perf_counter(), 0
    window, reached = [], None
    for ep in range(epochs):
        if ep in LR_DROPS:
            for g in opt.param_groups:
                g["lr"] /= 10.0
        model.train()
        perm = torch.randperm(n, device=DEV)
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            out = model(Xtr[idx])
            loss = F.cross_entropy(out, ytr[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            updates += 1
            if stop_at is not None:
                window.append((out.argmax(1) != ytr[idx]).float().mean().item())
                if len(window) > 50:
                    window.pop(0)
                if reached is None and len(window) == 50 and 100.0 * np.mean(window) < stop_at:
                    reached = updates
                if reached is not None or (max_updates and updates >= max_updates):
                    break
        if log_rows is not None:
            log_rows.append({"cond": tag, "seed": seed, "epoch": ep + 1,
                             "train_err": err_pct(model, Xtr, ytr),
                             "test_err": err_pct(model, Xte, yte)})
        if reached is not None or (max_updates and updates >= max_updates):
            break
    sec = time.perf_counter() - t0
    out = {"updates": updates, "reached": reached, "seconds": sec,
           "train_err": err_pct(model, Xtr, ytr), "test_err": err_pct(model, Xte, yte)}
    if tag and stop_at is None:
        print("  %-18s seed %d · 시험 %.2f%% · 학습 %.2f%% · 간격 %.2f · %.0f 초"
              % (tag, seed, out["test_err"], out["train_err"],
                 out["test_err"] - out["train_err"], sec))
    return model, out

### 구현이 맞는지 먼저 확인한다**셋을 본다.** (가) 내 LRN 이 식대로인가 — 작은 텐서에서 **직접 이중 반복으로 센 값**과 맞춰 본다. (나) PyTorch 내장 `nn.LocalResponseNorm` 과 얼마나 다른가($\alpha/n$ 규약). (다) 겹치는 풀링과 안 겹치는 풀링의 **출력 크기가 같은가** — 논문이 견준 조건이 그것이다.

In [ ]:
# ── 검증 ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
x = torch.randn(2, 8, 4, 4)

# (가) 식 그대로 직접 세기
n, k, alpha, beta = 5, 2.0, 1e-4, 0.75
want = torch.empty_like(x)
for b in range(x.shape[0]):
    for i in range(x.shape[1]):
        lo, hi = max(0, i - n // 2), min(x.shape[1] - 1, i + n // 2)
        s = sum(x[b, j] ** 2 for j in range(lo, hi + 1))
        want[b, i] = x[b, i] / (k + alpha * s) ** beta
got = PaperLRN(n, k, alpha, beta)(x)
print("(가) 내 LRN 과 직접 센 값의 최대 차이: %.3e" % (got - want).abs().max().item())
assert (got - want).abs().max().item() < 1e-5, "LRN 구현이 식과 다르다 — 여기서 멈춘다"

# (나) PyTorch 내장과의 차이
builtin = nn.LocalResponseNorm(size=n, alpha=alpha, beta=beta, k=k)(x)
fixed   = nn.LocalResponseNorm(size=n, alpha=alpha * n, beta=beta, k=k)(x)
print("(나) 내장(alpha 그대로)과의 최대 차이 %.3e · 내장(alpha x n)과의 최대 차이 %.3e"
      % ((got - builtin).abs().max().item(), (got - fixed).abs().max().item()))
print("    -> 내장은 alpha 를 n 으로 나눠 쓴다. 논문 값을 그대로 넣으면 다섯 배 약해진다.")

# (다) 두 풀링의 출력 크기
for z in (3, 2):
    m = Net(pool_z=z)
    with torch.no_grad():
        shp = tuple(m(torch.zeros(1, 3, 32, 32)).shape)
    p = sum(q.numel() for q in m.parameters())
    print("(다) z=%d · 출력 %s · 학습되는 값 %s" % (z, shp, format(p, ",")))

m = Net()
print()
print("층별 학습되는 값:", [(nm, q.numel()) for nm, q in m.named_parameters() if q.dim() > 1])
print("합계 %s" % format(sum(q.numel() for q in m.parameters()), ","))

## A. ReLU 가 tanh 보다 몇 배 빠른가**노트 13절 일곱이 가리키는 자리다.** 논문 그림 1 의 조건을 그대로 따른다.- 기준점은 **학습 오차 25%** (달리는 창 50 묶음).- **정칙화를 전혀 걸지 않는다** — 가중치 감쇠 0, 드롭아웃 없음, LRN 없음.- **두 비선형의 학습률을 각각 따로 고른다.** 먼저 작은 격자에서 seed 하나로 가장 빨리 닿는 학습률을 고르고, 그 값으로 seed 셋을 돌린다.

In [ ]:
# ── A. 학습률을 각각 따로 고른다 ────────────────────────────────────────
A_CFG = dict(lrn=False, pool_z=3, dropout=False)
GRID = [0.003, 0.01, 0.03, 0.1]
MAXU = 3000

print("[A] 학습률 고르기 (seed 0, 학습 오차 25% 에 닿기까지)")
best, rowsA_grid = {}, []
for act in ("relu", "tanh"):
    for lr in GRID:
        _, o = train(dict(A_CFG, act=act), seed=0, epochs=EPOCHS, lr=lr, wdecay=0.0,
                     stop_at=25.0, max_updates=MAXU)
        rowsA_grid.append({"act": act, "lr": lr, "updates_to_25": o["reached"],
                           "train_err_end": o["train_err"]})
        print("  %-5s lr %.3f -> %s" % (act, lr,
              ("갱신 %d 번" % o["reached"]) if o["reached"] else
              "닿지 못했다 (%d 번까지, 학습 오차 %.1f%%)" % (o["updates"], o["train_err"])))
    cand = [r for r in rowsA_grid if r["act"] == act and r["updates_to_25"]]
    best[act] = min(cand, key=lambda r: r["updates_to_25"])["lr"] if cand else None
print("  고른 학습률:", best)
save(pd.DataFrame(rowsA_grid), "A_grid.csv")

In [ ]:
# ── A. 고른 학습률로 seed 셋 ────────────────────────────────────────────
rowsA = []
print("[A] seed 셋")
for act in ("relu", "tanh"):
    if best[act] is None:
        print("  %-5s 격자 안에서 25%% 에 닿은 학습률이 없다 — 이 줄은 「닿지 못했다」로 적는다" % act)
        continue
    for s in SEEDS:
        _, o = train(dict(A_CFG, act=act), seed=s, epochs=EPOCHS, lr=best[act], wdecay=0.0,
                     stop_at=25.0, max_updates=MAXU)
        rowsA.append({"act": act, "lr": best[act], "seed": s,
                      "updates_to_25": o["reached"], "train_err_end": o["train_err"],
                      "seconds": o["seconds"]})
        print("  %-5s seed %d -> %s" % (act, s,
              ("갱신 %d 번" % o["reached"]) if o["reached"] else "닿지 못했다"))
dfA = pd.DataFrame(rowsA)
if len(dfA) and dfA.updates_to_25.notna().all():
    r_med = dfA[dfA.act == "relu"].updates_to_25.median()
    t_med = dfA[dfA.act == "tanh"].updates_to_25.median()
    print()
    print("  ReLU %d 번 (%s) · tanh %d 번 (%s) · 배수 %.2f"
          % (r_med, span(dfA[dfA.act == "relu"].updates_to_25),
             t_med, span(dfA[dfA.act == "tanh"].updates_to_25), t_med / r_med))
dfA = save(dfA, "A_relu_vs_tanh.csv")

## B · C · D · E. 넷을 하나씩 끄고, 함께도 끈다일곱 조건을 같은 데이터 · 같은 최적화 · 같은 바퀴 수로 돌린다.| 조건 | 무엇이 다른가 | 어느 실험 ||---|---|---|| 기준 (넷 다 켬) | — | 모두의 기준선 || LRN 끔 | 국소 반응 정규화를 뺀다 | B || 겹침 끔 ($z{=}2$) | 풀링 창을 2 로 (출력 크기는 같다) | C || 드롭아웃 끔 | 완전연결 앞 0.5 를 뺀다 | D || ReLU 끔 (tanh) | 비선형만 바꾼다 | A 의 짝 || **넷 다 끔** | 위 넷을 한꺼번에 | **E** || 초기화 논문 값 | 가중치를 $\sigma{=}0.01$ 로 (논문 5절) | 내가 바꾼 것을 재는 행 |**바퀴마다 학습 오차와 시험 오차를 남긴다** — D 의 「수렴이 느려지는가」와 곡선 그림에 쓴다.

In [ ]:
# ── B·C·D·E. 제거 실험 ──────────────────────────────────────────────────
rowsX, curves = [], []
print("[B·C·D·E] 일곱 조건 x seed 셋")
for name, cfg in CONDS.items():
    for s in SEEDS:
        _, o = train(cfg, seed=s, log_rows=curves, tag=name)
        rowsX.append({"cond": name, "seed": s, "test_err": o["test_err"],
                      "train_err": o["train_err"],
                      "gap": o["test_err"] - o["train_err"], "seconds": o["seconds"]})
dfX = pd.DataFrame(rowsX)
dfC = pd.DataFrame(curves)

print()
base = dfX[dfX.cond == "기준 (넷 다 켬)"].test_err
print("  기준 %.2f%% (%s) · 학습 %.2f%% · 간격 %.2f"
      % (base.median(), span(base),
         dfX[dfX.cond == "기준 (넷 다 켬)"].train_err.median(),
         dfX[dfX.cond == "기준 (넷 다 켬)"].gap.median()))
for name in CONDS:
    if name == "기준 (넷 다 켬)":
        continue
    v = dfX[dfX.cond == name].test_err
    widest = max(base.max() - base.min(), v.max() - v.min())
    print("  %-16s %.2f%% (%s) · 기준 대비 %+.2f · 변동 폭 %.2f -> %s"
          % (name, v.median(), span(v), v.median() - base.median(), widest,
             "폭보다 크다" if abs(v.median() - base.median()) > widest else "폭 안이다"))
dfX = save(dfX, "X_ablation.csv")
dfC = save(dfC, "X_curves.csv")

In [ ]:
# ── 그림 ────────────────────────────────────────────────────────────────
# 그림 안의 글자는 영어로 둔다 — Colab 에 한글 글꼴이 없어 네모로 깨진다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

EN = {"기준 (넷 다 켬)": "all four on", "LRN 끔": "no LRN", "겹침 끔 (z=2)": "no overlap (z=2)",
      "드롭아웃 끔": "no dropout", "ReLU 끔 (tanh)": "tanh instead of ReLU", "넷 다 끔": "all four off", "초기화 논문 값": "paper init (sigma=0.01)"}

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.2))

ax = axes[0]
if len(dfA) and dfA.updates_to_25.notna().all():
    vals = [dfA[dfA.act == a].updates_to_25.median() for a in ("relu", "tanh")]
    errs = [[vals[i] - dfA[dfA.act == a].updates_to_25.min(),
             dfA[dfA.act == a].updates_to_25.max() - vals[i]] for i, a in enumerate(("relu", "tanh"))]
    ax.bar(["ReLU", "tanh"], vals, 0.5, yerr=np.array(errs).T, capsize=4,
           color=["#B4552B", "#2E6B8A"])
    ax.set_ylabel("updates to reach 25% training error")
    ax.set_title("A. Lower is faster (paper: 6x on CIFAR-10)")
else:
    ax.text(0.5, 0.5, "A: not reached", ha="center")
ax.grid(alpha=.3, axis="y")

ax = axes[1]
names = list(CONDS.keys())
vals, errs = [], []
for n_ in names:
    v = dfX[dfX.cond == n_].test_err.values
    vals.append(np.median(v))
    errs.append([np.median(v) - v.min(), v.max() - np.median(v)])
ax.barh([EN[n_] for n_ in names][::-1], vals[::-1], 0.6,
        xerr=np.array(errs).T[:, ::-1], capsize=3, color="#2E6B8A")
ax.set_xlabel("test error (%)")
ax.set_title("B-E. Bars show the min-max span over seeds")
ax.grid(alpha=.3, axis="x")

ax = axes[2]
for n_ in ("기준 (넷 다 켬)", "드롭아웃 끔", "넷 다 끔"):
    piv = dfC[dfC.cond == n_].groupby("epoch")[["train_err", "test_err"]].median()
    line, = ax.plot(piv.index, piv.test_err, "-o", ms=3, label="%s (test)" % EN[n_])
    ax.plot(piv.index, piv.train_err, "--", lw=1.0, color=line.get_color(),
            label="%s (train)" % EN[n_])
ax.set_xlabel("epoch")
ax.set_ylabel("error (%)")
ax.set_title("D. The gap between train and test")
ax.grid(alpha=.3)
ax.legend(fontsize=7)

fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "ABCDE_summary.png"), dpi=140)
plt.close(fig)
print("저장: figures/ABCDE_summary.png")

## 요약 — 아래 출력을 그대로 `README.md` 의 결과 칸에 옮긴다**가설마다 맞았는지 틀렸는지를 적는다.** 틀렸으면 틀렸다고 적는다. **변동 폭보다 작은 차이로는 순서를 매기지 않는다.**

In [ ]:
# ── 요약 ────────────────────────────────────────────────────────────────
print("=" * 92)
print("AlexNet (2012) 실험 요약 · run_id %s" % RUN_ID)
print("설정: CIFAR-10 학습 %d · 시험 %d · %d 바퀴 · 묶음 %d · seed %s · 장치 %s"
      % (N_TRAIN, N_TEST, EPOCHS, BATCH, SEEDS, ENV["device"]))
print("주의: 데이터가 ImageNet 이 아니고 논문의 CIFAR-10 망 구조는 논문에 없다.")
print("      그래서 -1.7 / -1.4 / -0.4 나 13% / 11% 와 같은 칸에 놓지 않는다.")
print("=" * 92)

print()
print("[A] ReLU 대 tanh — H1")
if len(dfA) and dfA.updates_to_25.notna().all():
    r = dfA[dfA.act == "relu"].updates_to_25
    t = dfA[dfA.act == "tanh"].updates_to_25
    print("  고른 학습률: ReLU %.3f · tanh %.3f (각각 따로 골랐다 — 논문 그림 1 의 조건)"
          % (best["relu"], best["tanh"]))
    print("  ReLU 갱신 %d 번 (%s) · tanh %d 번 (%s)"
          % (r.median(), span(r), t.median(), span(t)))
    print("  배수 %.2f (논문 그림 1 은 6 배, 다른 망 · 다른 학습률)" % (t.median() / r.median()))
    print("  H1 판정:", "맞았다 — ReLU 쪽이 빨리 닿았다" if t.median() > r.median() else
          "틀렸다 — tanh 쪽이 같거나 빨랐다. 이 망에서는 포화가 문제가 되지 않았다는 뜻이다")
else:
    print("  25% 에 닿지 못한 조건이 있다 — 「닿지 못했다」로 적고 배수를 내지 않는다.")

base = dfX[dfX.cond == "기준 (넷 다 켬)"]
b_med, b_span = base.test_err.median(), span(base.test_err)
deltas = {}
print()
print("[B·C·D] 하나씩 껐을 때 — H2 · H3 · H4")
print("  기준 (넷 다 켬) %.2f%% (%s) · 학습 %.2f%% · 간격 %.2f"
      % (b_med, b_span, base.train_err.median(), base.gap.median()))
for name, h in [("LRN 끔", "H2"), ("겹침 끔 (z=2)", "H3"), ("드롭아웃 끔", "H4")]:
    v = dfX[dfX.cond == name]
    d = v.test_err.median() - b_med
    w = max(base.test_err.max() - base.test_err.min(), v.test_err.max() - v.test_err.min())
    deltas[name] = d
    print("  %-14s %.2f%% (%s) · 차이 %+.2f · 변동 폭 %.2f · 간격 %.2f -> %s"
          % (name, v.test_err.median(), span(v.test_err), d, w, v.gap.median(),
             "폭보다 크다" if abs(d) > w else "폭 안이다"))
    print("     %s 판정: %s" % (h, "켜는 쪽이 낫다 (차이가 변동 폭보다 크다)" if d > w else
                              ("끄는 쪽이 낫다 (차이가 변동 폭보다 크다)" if -d > w else
                               "차이가 변동 폭 안이라 순서를 매기지 않는다")))
v = dfX[dfX.cond == "드롭아웃 끔"]
print("  D 를 읽는 법: 드롭아웃을 끄면 간격이 %.2f -> %.2f 로 간다."
      % (base.gap.median(), v.gap.median()))
print("     간격이 늘었는데 시험 오차가 안 올랐다면, 이 규모에서 과적합이 아직 값을 갉지 않은 것이다.")

name = "ReLU 끔 (tanh)"
v = dfX[dfX.cond == name]
deltas[name] = v.test_err.median() - b_med
print("  %-14s %.2f%% (%s) · 차이 %+.2f" % (name, v.test_err.median(), span(v.test_err), deltas[name]))

name = "초기화 논문 값"
v = dfX[dfX.cond == name]
w = max(base.test_err.max() - base.test_err.min(), v.test_err.max() - v.test_err.min())
print("  %-14s %.2f%% (%s) · 차이 %+.2f · 변동 폭 %.2f -> %s  [내가 바꾼 것을 잰 행. E 의 합에는 넣지 않는다]"
      % (name, v.test_err.median(), span(v.test_err), v.test_err.median() - b_med, w,
         "폭보다 크다" if abs(v.test_err.median() - b_med) > w else "폭 안이다"))

print()
print("[E] 넷을 함께 껐을 때 — H5  (논문에 없는 행)")
allofff = dfX[dfX.cond == "넷 다 끔"]
d_all = allofff.test_err.median() - b_med
d_sum = sum(deltas.values())
w = max(base.test_err.max() - base.test_err.min(),
        allofff.test_err.max() - allofff.test_err.min())
print("  넷 다 끔 %.2f%% (%s) · 기준 대비 %+.2f" % (allofff.test_err.median(), span(allofff.test_err), d_all))
print("  하나씩 끈 차이의 합 %+.2f (LRN %+.2f · 겹침 %+.2f · 드롭아웃 %+.2f · ReLU %+.2f)"
      % (d_sum, deltas["LRN 끔"], deltas["겹침 끔 (z=2)"], deltas["드롭아웃 끔"], deltas["ReLU 끔 (tanh)"]))
print("  둘의 차이 %+.2f · 변동 폭 %.2f" % (d_all - d_sum, w))
if abs(d_all - d_sum) <= w:
    print("  H5 판정: 틀렸다 — 합과 함께 끈 것이 변동 폭 안에서 같다.")
    print("           그러면 논문처럼 따로 적고 더해 읽어도 되는 것이고, 그것도 답이다.")
elif d_all > d_sum:
    print("  H5 판정: 맞았다 — 함께 끈 손해가 합보다 크다. 곧 넷이 서로를 돕고 있었다.")
else:
    print("  H5 판정: 반대 방향으로 맞았다 — 함께 끈 손해가 합보다 작다. 넷의 이득이 서로 겹친다.")

print()
_a_sec = float(dfA["seconds"].sum()) if "seconds" in dfA.columns else 0.0
print("시간: 제거 실험 한 칸에 %.0f~%.0f 초 · 전부 더해 %.0f 초"
      % (dfX.seconds.min(), dfX.seconds.max(), dfX.seconds.sum() + _a_sec))
print("=" * 92)